# Modules

In [ ]:
from Utils import Timer, Duur, meet_duur,toon_duur, convert_wkt_string_to_gps, batch_transform_geometry_to_gps_string, convert_coords_to_gps

from scipy.spatial.distance import cdist
from IPython.display import display
import geopandas as gpd
from pyproj import Transformer
from shapely import wkt
from shapely.geometry.base import BaseGeometry
import re
from scipy.spatial import cKDTree
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)       # Geen omvouwing naar volgende regel
pd.set_option('display.float_format', '{:.6f}'.format)  # 6 decimalen voor floats
pd.set_option('display.max_seq_items', None)
pd.set_option('display.precision', 10) 
pd.set_option('display.show_dimensions', True)
import numpy as np
np.set_printoptions(threshold=np.inf, linewidth=np.inf)

# Inweva

In [ ]:
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, \
{df.shape[1]} kolommen')

In [ ]:
JAAR = 2025
INWEVA_DIR = Path("Data/Inweva")
INWEVA_DIR.mkdir(parents=True, exist_ok=True)
INWEVA_ZIP = INWEVA_DIR / f'INWEVA_{JAAR}.zip'
INWEVA_EXTRACTED = INWEVA_DIR / f'INWEVA_{JAAR}'
URL = f'https://downloads.rijkswaterstaatdata.nl/inweva/INWEVA_{JAAR}.zip'


def download_inweva():
    if not INWEVA_ZIP.exists():
        with Duur('Download INWEVA'):
            urllib.request.urlretrieve(URL, INWEVA_ZIP)
    if not INWEVA_EXTRACTED.exists():
        with zipfile.ZipFile(INWEVA_ZIP) as z:
            z.extractall(INWEVA_DIR)


def lees_netwerk():
    shp_pad = INWEVA_EXTRACTED / f'INWEVA_{JAAR}_netwerk' / f'INWEVA{JAAR}.shp'
    with Duur('Inlezen INWEVA shapefile'):
        netwerk = gpd.read_file(shp_pad)
    netwerk['VBN_ID'] = netwerk['VBN_ID'].astype(str)
    return netwerk


def lees_werkdag():
    return pd.read_csv(INWEVA_EXTRACTED / f'inweva_werkdag_{JAAR}DEC.csv',
                       sep=';', dtype={'VBN_ID': str})


def bouw_inweva_per_wvk(netwerk, werkdag):
    inweva = netwerk.merge(
        werkdag[['VBN_ID', 'AL_E_WR', 'L1_E_WR', 'L2_E_WR', 'L3_E_WR']],
        on='VBN_ID', how='left'
    )
    inweva_wvk = (
        inweva
        .assign(WVK_ID=inweva['WVK_IDS'].fillna('').str.split(','))
        .explode('WVK_ID')
    )
    inweva_wvk['WVK_ID'] = inweva_wvk['WVK_ID'].str.strip()
    inweva_wvk = inweva_wvk[inweva_wvk['WVK_ID'] != '']
    inweva_wvk['wvk_id'] = pd.to_numeric(inweva_wvk['WVK_ID'], errors='coerce').astype('Int64')

    per_wvk = (
        inweva_wvk[['wvk_id', 'L1_E_WR', 'L2_E_WR', 'L3_E_WR', 'AL_E_WR', 'VBNOMSTXT']]
        .drop_duplicates(subset='wvk_id', keep='first')
        .reset_index(drop=True)
        .rename(columns={
            'L1_E_WR': 'klein_voertuig',
            'L2_E_WR': 'middel_voertuig',
            'L3_E_WR': 'lang_voertuig',
            'AL_E_WR': 'totaal_voertuig',
            'VBNOMSTXT': '_inweva_omschrijving',
        })
    )
    voertuig_cols = ['klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig']
    per_wvk[voertuig_cols] = per_wvk[voertuig_cols].fillna(0).round().astype('Int64')
    per_wvk['wvk_id'] = per_wvk['wvk_id'].astype('Int64')
    return per_wvk


def koppel_inweva(df, inweva_per_wvk):
    n_voor = len(df)
    voertuig_cols = ['klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig',
                     '_inweva_omschrijving']
    df = df.drop(columns=[c for c in voertuig_cols if c in df.columns])
    df['wvk_id'] = pd.to_numeric(df['wvk_id'], errors='coerce').astype('Int64')
    df = df.merge(inweva_per_wvk, on='wvk_id', how='left')
    assert len(df) == n_voor, f'Hectopunten omvang VERANDERD! {n_voor} -> {len(df)}'
    return df


download_inweva()
netwerk = lees_netwerk()
werkdag = lees_werkdag()
inweva_per_wvk = bouw_inweva_per_wvk(netwerk, werkdag)

df_voor = len(df)
df = koppel_inweva(df, inweva_per_wvk)
df = df.loc[:, ~df.columns.duplicated()].copy()

# VBNOMSTXT als steekwoord/notitie aan info plakken
df['info'] = df['info'].fillna('').astype(str)
omschr = df['_inweva_omschrijving'].fillna('').astype(str)
df['info'] = df['info'] + omschr.where(omschr == '', ' | ' + omschr).where(df['info'] != '', omschr)
df = df.drop(columns=['_inweva_omschrijving'])

cols = [
    'wvk_id', 'wegnr_hmp', 'Zijde', 'hectomtrng', 'hecto_lttr',
    'klein_voertuig', 'middel_voertuig', 'lang_voertuig',
    'totaal_voertuig', 'distrnaam', 'info', 'streetsmart_link', 'google_maps_link', '|', 'wegbehnaam',
    'beginkm', 'eindkm', 'snelwegnummer', 'wegnummer', 'wegnr_aw', 'rijrichtng', 'oplopend', 'afstand',
    'hectopunt_geometry', 'wegvak_geometry',
    'gps_coordinaten', 'hectopunt_rd_coordinaten',
]
df = df[cols]
df = df.loc[:, ~df.columns.duplicated()].copy()

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, {df.shape[1]} kolommen')
n_match = df['totaal_voertuig'].notna().sum()
print(f'Hectopunten voor merge : {df_voor:,}')
print(f'Hectopunten na merge   : {df.shape[0]:,}')
print(f'Hectopunten mét INWEVA : {n_match:,} ({n_match/len(df)*100:.1f}%)')
print(f'Kolommen na merge ({len(df.columns)}): {df.columns.tolist()}')

# Bochten

In [ ]:
bochten = gpd.read_file("Data/Bochten/Bochten/bochten_w.shp")
bochten.rename(columns={'WVK_ID': 'wvk_id'}, inplace=True)
bochten['wvk_id'] = bochten['wvk_id'].astype(float)
print(f'Bochten geladen: {bochten.shape[0]:,} rijen, {bochten.shape[1]} kolommen')

hecto_wvk   = {int(x) for x in df['wvk_id'].dropna().tolist()}
bocht_wvk   = {int(x) for x in bochten['wvk_id'].dropna().tolist()}
overlap_wvk = hecto_wvk & bocht_wvk
print(f'Hectopunten (wvk_ids): {len(hecto_wvk):,}')
print(f'Bochten     (wvk_ids): {len(bocht_wvk):,}')

_hecto_voor   = len(df)
_bochten_voor = len(bochten)

transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)


def _coords(geom):
    if geom is None:
        return None
    if geom.geom_type == 'LineString':
        return list(geom.coords)
    if geom.geom_type == 'MultiLineString':
        return [c for line in geom.geoms for c in line.coords]
    return None


def rd_coords_flat(geom):
    coords = _coords(geom)
    if coords is None:
        return None
    return ", ".join(f"{round(x,3)}, {round(y,3)}" for x, y in coords)


def gps_coords_flat(geom):
    coords = _coords(geom)
    if coords is None:
        return None
    return ", ".join(
        f"{round(lat,6)}, {round(lon,6)}"
        for x, y in coords
        for lon, lat in [transformer.transform(x, y)]
    )


df['hectopunt_geometry_flat'] = df['hectopunt_geometry'].apply(
    lambda g: f"{repr(g.geoms[0].x)}, {repr(g.geoms[0].y)}"
)
bochten['bochten_geometry_flat']   = bochten['geometry'].apply(
    lambda g: ', '.join(f"{x}, {y}" for x, y in g.coords)
)
bochten['bochten_rd_coordinaten']  = bochten['geometry'].apply(rd_coords_flat)
bochten['bochten_gps_coordinaten'] = bochten['geometry'].apply(gps_coords_flat)

from scipy.spatial.distance import cdist
from collections import defaultdict


def parse_first_rd_point(geom_str):
    parts = str(geom_str).split(',')
    return float(parts[0].strip()), float(parts[1].strip())


def parse_middle_rd_point(geom_str):
    parts  = [float(p.strip()) for p in str(geom_str).split(',')]
    coords = [(parts[i], parts[i + 1]) for i in range(0, len(parts) - 1, 2)]
    return coords[len(coords) // 2]


df['_rd_x'], df['_rd_y'] = zip(*df['hectopunt_geometry_flat'].apply(parse_first_rd_point))
bochten['_rd_x'], bochten['_rd_y'] = zip(*bochten['bochten_geometry_flat'].apply(parse_middle_rd_point))

df = df.reset_index(drop=True)

# 1) per bocht: dichtstbijzijnde hectopunt binnen wvk_id, max 250 m
hecto_per_wvk = {wvk: sub for wvk, sub in df.groupby('wvk_id')}

bocht_naar_hecto = {}
for bocht_idx, brow in bochten.iterrows():
    sub = hecto_per_wvk.get(brow['wvk_id'])
    if sub is None or sub.empty:
        continue
    bpt       = np.array([[brow['_rd_x'], brow['_rd_y']]])
    hpts      = sub[['_rd_x', '_rd_y']].values
    distances = cdist(bpt, hpts)[0]
    nearest   = np.argmin(distances)
    if distances[nearest] <= 250:
        bocht_naar_hecto[bocht_idx] = sub.index[nearest]

# 2) per hectopunt: alle bochten verzamelen
hecto_naar_bochten = defaultdict(list)
for bocht_idx, hecto_pos in bocht_naar_hecto.items():
    hecto_naar_bochten[hecto_pos].append(bocht_idx)

# 3) per hectopunt: comma-strings opbouwen
def _join(idxs, kol):
    return ', '.join(str(bochten.loc[i, kol]) for i in idxs)


draaihoek_kol, boogstraal_kol     = [], []
geom_flat_kol, rd_kol, gps_kol    = [], [], []
check_lijn_kol, aantal_kol        = [], []

for pos in range(len(df)):
    idxs = hecto_naar_bochten.get(pos, [])
    aantal_kol.append(len(idxs))
    if not idxs:
        draaihoek_kol.append(None);  boogstraal_kol.append(None)
        geom_flat_kol.append(None);  rd_kol.append(None)
        gps_kol.append(None);        check_lijn_kol.append(None)
        continue
    draaihoek_kol .append(_join(idxs, 'DRAAIHOEK'))
    boogstraal_kol.append(_join(idxs, 'BOOGSTRAAL'))
    geom_flat_kol .append(_join(idxs, 'bochten_geometry_flat'))
    rd_kol        .append(_join(idxs, 'bochten_rd_coordinaten'))
    gps_kol       .append(_join(idxs, 'bochten_gps_coordinaten'))
    hx, hy = df.loc[pos, '_rd_x'], df.loc[pos, '_rd_y']
    check_lijn_kol.append(', '.join(
        f"LINESTRING ({hx} {hy}, {bochten.loc[i, '_rd_x']} {bochten.loc[i, '_rd_y']})"
        for i in idxs
    ))

df['draaihoek']               = draaihoek_kol
df['boogstraal']              = boogstraal_kol
df['bochten_geometry_flat']   = geom_flat_kol
df['bochten_rd_coordinaten']  = rd_kol
df['bochten_gps_coordinaten'] = gps_kol
df['aantal_bochten']          = aantal_kol
df['check_koppeling_lijn']    = check_lijn_kol

df.drop(columns=['_rd_x', '_rd_y'], inplace=True)
bochten.drop(columns=['_rd_x', '_rd_y'], inplace=True)
df = df.sort_values('aantal_bochten', ascending=False).reset_index(drop=True)
df = df[['wvk_id', 'wegnr_hmp', 'Zijde', 'hectomtrng', 'hecto_lttr',
         'draaihoek', 'boogstraal',
         'klein_voertuig', 'middel_voertuig', 'lang_voertuig', 'totaal_voertuig',
         'distrnaam', 'info', 'streetsmart_link', 'google_maps_link', '|',
         'wegbehnaam', 'beginkm', 'eindkm', 'snelwegnummer', 'wegnummer',
         'wegnr_aw', 'rijrichtng', 'oplopend', 'afstand',
         'hectopunt_geometry', 'wegvak_geometry', 'gps_coordinaten',
         'hectopunt_rd_coordinaten', 'hectopunt_geometry_flat',
         'bochten_geometry_flat', 'bochten_rd_coordinaten', 'bochten_gps_coordinaten',
         'aantal_bochten', 'check_koppeling_lijn']]
df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, \
{df.shape[1]} kolommen')

# 4) diagnostiek
hecto_match   = sum(1 for v in aantal_kol if v > 0)
hecto_geen    = len(df) - hecto_match
bochten_match = len(bocht_naar_hecto)
bochten_geen  = _bochten_voor - bochten_match
max_per_hecto = max(aantal_kol) if aantal_kol else 0

print(f'── Hectopunten ──────────────────────────────')
print(f'  Voor koppeling   : {_hecto_voor:,}')
print(f'  Na koppeling     : {len(df):,}')
print(f'  Met bocht(en)    : {hecto_match:,}  ({hecto_match/len(df)*100:.1f}%)')
print(f'  Zonder bocht     : {hecto_geen:,}  ({hecto_geen/len(df)*100:.1f}%)')
print(f'  Max bochten/hecto: {max_per_hecto}')
print(f'── Bochten ──────────────────────────────────')
print(f'  Voor koppeling   : {_bochten_voor:,}')
print(f'  Gekoppeld        : {bochten_match:,}  ({bochten_match/_bochten_voor*100:.1f}%)')
print(f'  Ongekoppeld      : {bochten_geen:,}  ({bochten_geen/_bochten_voor*100:.1f}%)')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')
# display(df.head(1).T)

# Inspectigence

In [ ]:
PAD_2023 = "Data/Inspectigence/Inspectigence-2023/csv/Inspectigence_Part04_2023.csv"
PAD_2025 = "Data/Inspectigence/Inspectigence-2025/csv/Inspectigence_Part04_2025.csv"

transformer_to_rd = Transformer.from_crs('EPSG:4326', 'EPSG:28992', always_xy=True)


def lees_inspectigence(pad):
    ins = pd.read_csv(pad)
    ins = ins.rename(columns={'Health_Score': 'health', 'Visibility_Score': 'visibility'})
    ins = ins.drop_duplicates(subset=['KeyID'], keep='first').reset_index(drop=True)
    return ins[['KeyID', 'Geometry', 'health', 'visibility', 'BENAMING']]


def parse_geom(g):
    return g if isinstance(g, BaseGeometry) else wkt.loads(g)


def linestring_xy_rd(geom):
    coords = list(geom.coords)
    lon, lat = coords[len(coords) // 2]
    return transformer_to_rd.transform(lon, lat)


def parse_hectopunt_rd(geom):
    pt = geom.geoms[0] if hasattr(geom, 'geoms') else geom
    return pt.x, pt.y


def sorteer_csv(s, ascending=True):
    if pd.isna(s):
        return s
    vals = sorted((float(v.strip()) for v in str(s).split(',') if v.strip()),
                  reverse=not ascending)
    return ', '.join(f'{v}' for v in vals)


def eerste_waarde(s, default):
    return float(str(s).split(',')[0]) if pd.notna(s) else default


def koppel_inspectigence(df, ins, jaar):
    ins = ins.copy()
    ins['Geometry'] = ins['Geometry'].apply(parse_geom)

    ins_xy   = np.array([linestring_xy_rd(g) for g in ins['Geometry']])
    hecto_xy = np.array(df['hectopunt_geometry'].apply(parse_hectopunt_rd).tolist())

    tree = cKDTree(hecto_xy)
    _, indices = tree.query(ins_xy, k=1)

    ins_m = pd.DataFrame({
        '_hecto_idx': indices,
        'health':     ins['health'].values,
        'visibility': ins['visibility'].values,
        'BENAMING':   ins['BENAMING'].values,
    })

    join_asc       = lambda s: ', '.join(f'{v}' for v in sorted(float(x) for x in s.dropna().tolist()))
    join_benaming  = lambda s: ' | '.join(sorted(set(s.dropna().astype(str).tolist())))

    ins_grouped = (
        ins_m
        .groupby('_hecto_idx', as_index=False)
        .agg({'health': join_asc, 'visibility': join_asc, 'BENAMING': join_benaming})
    )
    ins_grouped[f'_n_{jaar}'] = ins_m.groupby('_hecto_idx').size().values
    ins_grouped = ins_grouped.rename(columns={
        'health':     f'health_{jaar}',
        'visibility': f'visibility_{jaar}',
        'BENAMING':   f'_benaming_{jaar}',
    })

    df = df.reset_index(drop=True)
    df['_hecto_idx'] = df.index
    df = df.merge(ins_grouped, on='_hecto_idx', how='left').drop(columns=['_hecto_idx'])
    return df


def voeg_benaming_aan_info(df):
    df['info'] = df['info'].fillna('').astype(str)
    samen = []
    for ben23, ben25 in zip(df['_benaming_2023'].fillna(''), df['_benaming_2025'].fillna('')):
        delen = [b for b in (ben23, ben25) if b]
        samen.append(' | '.join(sorted(set(' | '.join(delen).split(' | ')))) if delen else '')
    extra = pd.Series(samen, index=df.index)
    df['info'] = np.where(
        extra == '', df['info'],
        np.where(df['info'] == '', extra, df['info'] + ' | ' + extra)
    )
    return df.drop(columns=['_benaming_2023', '_benaming_2025'])


def diagnose(df, ins, jaar):
    n_h     = len(df)
    n_match = df[f'health_{jaar}'].notna().sum()
    print(f'── Inspectigence {jaar} ──────────────────────')
    print(f'  Hectopunten              : {n_h:,}')
    print(f'  Inspectigence-records    : {len(ins):,}')
    print(f'  Hectopunten met match    : {n_match:,} ({n_match/n_h*100:.1f}%)')
    print(f'  Hectopunten zonder match : {n_h - n_match:,} ({(n_h - n_match)/n_h*100:.1f}%)')


# 0) Schone start
oud = [c for c in df.columns
       if c.startswith(('ins_', 'ins23_', 'ins25_', 'ins2023_', 'ins2025_'))
       or c in ('health_2023', 'visibility_2023', 'health_2025', 'visibility_2025',
                'aantal_inspecties', '_benaming_2023', '_benaming_2025')]
df = df.drop(columns=oud, errors='ignore')

# 1) Inlezen + koppelen
_hecto_voor = len(df)
ins_2023 = lees_inspectigence(PAD_2023)
ins_2025 = lees_inspectigence(PAD_2025)
print(f'Inspectigence 2023 : {len(ins_2023):,} records')
print(f'Inspectigence 2025 : {len(ins_2025):,} records')

df = koppel_inspectigence(df, ins_2023, jaar=2023)
df = koppel_inspectigence(df, ins_2025, jaar=2025)
assert len(df) == _hecto_voor, f'Hectopunten omvang VERANDERD! {_hecto_voor} -> {len(df)}'

# 2) BENAMING aan info plakken (uniek, ' | '-gescheiden)
df = voeg_benaming_aan_info(df)

# 3) aantal_inspecties = max over beide jaren
df['aantal_inspecties'] = df[['_n_2023', '_n_2025']].max(axis=1).astype('Int64')
df = df.drop(columns=['_n_2023', '_n_2025'])

# 4) Per cel intern sorteren zodat [0] altijd het juiste uiterste is
df['draaihoek']  = df['draaihoek'].apply(lambda s: sorteer_csv(s, ascending=False))
df['boogstraal'] = df['boogstraal'].apply(lambda s: sorteer_csv(s, ascending=True))

# 5) Kolomvolgorde: scores na boogstraal, aantal_inspecties achteraan
score_cols = ['health_2023', 'visibility_2023', 'health_2025', 'visibility_2025']
for i, c in enumerate(score_cols):
    df.insert(df.columns.get_loc('boogstraal') + 1 + i, c, df.pop(c))
df['aantal_inspecties'] = df.pop('aantal_inspecties')

# 6) Sorteren df: hoogste hoek boven, dan laagste visibility_2025
df = (df
      .assign(
          _max_hoek=df['draaihoek'].apply(lambda s: eerste_waarde(s, -np.inf)),
          _min_vis =df['visibility_2025'].apply(lambda s: eerste_waarde(s,  np.inf)),
      )
      .sort_values(['_max_hoek', '_min_vis'], ascending=[False, True])
      .drop(columns=['_max_hoek', '_min_vis'])
      .reset_index(drop=True))

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, {df.shape[1]} kolommen')

# 7) Diagnose
print()
diagnose(df, ins_2023, 2023)
print()
diagnose(df, ins_2025, 2025)
print()
print(f'Hectopunten voor koppeling : {_hecto_voor:,}')
print(f'Hectopunten na koppeling   : {len(df):,}')
print(f'Max inspecties per hecto   : {df["aantal_inspecties"].max()}')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')

# Deklagen

In [ ]:
DEKLAGEN_XLS = Path('Data/Deklagen/Deklagen_M26_NL_Totaal Definitief.xls')


def lees_deklagen():
    with Duur('Inlezen Deklagenlijst'):
        dek = pd.read_excel(DEKLAGEN_XLS, sheet_name='DeklagenM26Totaal')
    dek = dek.dropna(subset=['VAN', 'TOT']).copy()
    dek['AANLEGDATUM'] = pd.to_datetime(dek['AANLEGDATUM'], format='%d-%m-%Y', errors='coerce')
    return dek


def expand_deklagen(dek):
    dek = dek.copy()
    dek['km_int']    = dek.apply(lambda r: list(range(int(r['VAN']), int(r['TOT']) + 1)), axis=1)
    dek = dek.explode('km_int')
    dek['km_int']    = dek['km_int'].astype(int)
    dek['wegnr_hmp'] = dek['WEG'].astype(int).apply(lambda n: f'A{n}')
    return dek


def koppel_deklagen(df, dek_exp):
    matches = df.merge(
        dek_exp,
        left_on=['wegnr_hmp', 'hectomtrng'],
        right_on=['wegnr_hmp', 'km_int'],
        how='inner',
    )
    matches = matches.drop_duplicates(
        subset=['wvk_id', 'hectomtrng', 'BAAN', 'STROOK', 'VAN', 'TOT', 'AANLEGDATUM']
    )
    return matches.sort_values('AANLEGDATUM')


def aggregeer_per_hectopunt(matches):
    join_str = lambda s: ', '.join(s.dropna().astype(str).tolist())
    return (
        matches
        .groupby(['wvk_id', 'hectomtrng'], as_index=False)
        .agg(
            deklaagsoort    = ('DEKLAAGSOORT', join_str),
            aanlegdatum     = ('AANLEGDATUM',  lambda s: ', '.join(s.dt.strftime('%Y-%m-%d').fillna(''))),
            strook          = ('STROOK',       join_str),
            aantal_deklagen = ('DEKLAAGSOORT', 'size'),
        )
    )


def diagnose(df, dek, matches, dek_exp):
    n_h     = len(df)
    n_match = df['deklaagsoort'].notna().sum()
    wegen_in_dek = set(dek_exp['wegnr_hmp'])
    wegen_in_df  = set(df['wegnr_hmp'].dropna())
    ontbreekt    = wegen_in_df - wegen_in_dek
    print(f'── Deklagen ─────────────────────────────────')
    print(f'  Hectopunten              : {n_h:,}')
    print(f'  Deklagen-records         : {len(dek):,}')
    print(f'  Long matches             : {len(matches):,}')
    print(f'  Hectopunten met deklaag  : {n_match:,} ({n_match/n_h*100:.1f}%)')
    print(f'  Hectopunten zonder       : {n_h - n_match:,} ({(n_h - n_match)/n_h*100:.1f}%)')
    print(f'  Max deklagen / hecto     : {df["aantal_deklagen"].max()}')
    print(f'  Mediaan deklagen / hecto : {df["aantal_deklagen"].median()}')
    print(f'  Wegen in df              : {sorted(wegen_in_df)}')
    print(f'  Wegen in df NIET in dek  : {sorted(ontbreekt) if ontbreekt else "geen"}')


# 0) Schone start
df = df.drop(columns=['deklaagsoort', 'aanlegdatum', 'strook', 'aantal_deklagen'],
             errors='ignore')

# 1) Inlezen + voorbereiden
_hecto_voor = len(df)
dek      = lees_deklagen()
dek_exp  = expand_deklagen(dek)
print(f'Deklagen origineel       : {len(dek):,}')
print(f'Deklagen na km-expand    : {len(dek_exp):,}')

# 2) Koppelen (op wegnr_hmp + hectomtrng) en aggregeren
matches      = koppel_deklagen(df, dek_exp)
deklagen_hp  = aggregeer_per_hectopunt(matches)

# 3) Left-merge in df (omvang gelijk)
df = df.merge(deklagen_hp, on=['wvk_id', 'hectomtrng'], how='left')
assert len(df) == _hecto_voor, f'Hectopunten omvang VERANDERD! {_hecto_voor} -> {len(df)}'

# 4) Kolomvolgorde: deklaag-velden direct na visibility_2025, aantal_deklagen achteraan
nieuwe_cols = ['deklaagsoort', 'aanlegdatum', 'strook']
for i, c in enumerate(nieuwe_cols):
    df.insert(df.columns.get_loc('visibility_2025') + 1 + i, c, df.pop(c))
df['aantal_deklagen'] = df.pop('aantal_deklagen')

df.to_pickle("Output/Data_Analyse-Dataset.pkl")
df = pd.read_pickle("Output/Data_Analyse-Dataset.pkl")
print(f'Wegvakken geladen : {df.shape[0]:,} rijen, \
{df.shape[1]} kolommen')

# 5) Diagnose
print()
diagnose(df, dek, matches, dek_exp)
print()
print(f'Hectopunten voor koppeling : {_hecto_voor:,}')
print(f'Hectopunten na koppeling   : {len(df):,}')
print(f'Kolommen na koppeling ({len(df.columns)}): {df.columns.tolist()}')
# display(df.head(5).T)

In [ ]:
df.columns

In [ ]:
# df[['wvk_id', 'wegnr_hmp', 'Zijde', 'hectomtrng', 'hecto_lttr', 'draaihoek',
#        'boogstraal', 'health_2023', 'visibility_2023', 'health_2025',
#        'visibility_2025', 'klein_voertuig', 'middel_voertuig', 'lang_voertuig',
#        'totaal_voertuig', 'distrnaam', 'info', 'streetsmart_link',
#        'google_maps_link']].head(100).T

In [ ]:
# mask = df['visibility_2025'].apply(
#     lambda s: pd.notna(s) and 2 <= float(str(s).split(',')[0]) <= 4
# )
# df[mask].head(20).T

In [ ]:
df.columns

In [ ]:
# import matplotlib.pyplot as plt

# netwerk_plot = netwerk.merge(
#     werkdag[['VBN_ID', 'AL_E_WR', 'L1_E_WR', 'L2_E_WR', 'L3_E_WR']],
#     on='VBN_ID', how='left'
# ).rename(columns={
#     'L1_E_WR': 'klein_voertuig',
#     'L2_E_WR': 'middel_voertuig',
#     'L3_E_WR': 'lang_voertuig',
#     'AL_E_WR': 'totaal_voertuig',
# })

# categorieen = [
#     ('klein_voertuig',  'Klein voertuig (L1)'),
#     ('middel_voertuig', 'Middel voertuig (L2)'),
#     ('lang_voertuig',   'Lang voertuig (L3)'),
#     ('totaal_voertuig', 'Totaal voertuigen (AL)'),
# ]

# OUTPUT_DIR = Path('Output')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# fig, axes = plt.subplots(4, 1, figsize=(14, 32))
# for ax, (kol, titel) in zip(axes, categorieen):
#     netwerk_plot.plot(column=kol, ax=ax, legend=True,
#                       legend_kwds={'label': 'voertuigen/etmaal (werkdag)'},
#                       cmap='RdYlGn_r', linewidth=0.6)
#     ax.set_title(f'{titel} — etmaal werkdag ({JAAR})', fontweight='bold')
#     ax.set_axis_off()
# plt.tight_layout()

# uitvoer_pad = OUTPUT_DIR / f'inweva_netwerk_{JAAR}.png'
# fig.savefig(uitvoer_pad, dpi=150, bbox_inches='tight')
# print(f'Opgeslagen: {uitvoer_pad}')
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import geopandas as gpd

# OUTPUT_DIR = Path('Output')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# matches_uniq = matches.drop_duplicates(subset=['wvk_id', 'hectomtrng', 'AANLEGDATUM', 'STROOK'])

# per_weg = (
#     df.assign(_heeft=df['deklaagsoort'].notna())
#       .groupby('wegnr_hmp')
#       .agg(hectopunten=('wvk_id', 'size'),
#            met_deklaag=('_heeft', 'sum'))
# )
# per_weg['dekking_%'] = (per_weg['met_deklaag'] / per_weg['hectopunten'] * 100).round(1)
# per_weg = per_weg.sort_values('hectopunten', ascending=True)

# soort_cnt = matches_uniq['DEKLAAGSOORT'].value_counts()
# jaar_cnt  = matches_uniq['AANLEGDATUM'].dt.year.value_counts().sort_index()
# n_dek     = df['aantal_deklagen'].dropna().astype(int).value_counts().sort_index()


# def plot_per_snelweg(ax):
#     ax.barh(per_weg.index, per_weg['hectopunten'], color='#bdbdbd', label='Totaal hectopunten')
#     ax.barh(per_weg.index, per_weg['met_deklaag'], color='#2196F3', label='Met deklaag')
#     for i, (tot, mt, pct) in enumerate(zip(per_weg['hectopunten'],
#                                            per_weg['met_deklaag'],
#                                            per_weg['dekking_%'])):
#         ax.text(tot + max(per_weg['hectopunten'])*0.01, i,
#                 f'{mt:,}/{tot:,}  ({pct:.0f}%)', va='center', fontsize=14)
#     ax.set_xlabel('Aantal hectopunten', fontsize=14)
#     ax.set_title('Hectopunten met deklaag per snelweg', fontweight='bold', fontsize=16)
#     ax.legend(loc='lower right', fontsize=13)


# def plot_deklaagsoort(ax):
#     clrs = plt.cm.tab20(np.linspace(0, 1, len(soort_cnt)))
#     ax.barh(soort_cnt.index[::-1], soort_cnt.values[::-1], color=clrs)
#     for i, v in enumerate(soort_cnt.values[::-1]):
#         ax.text(v + max(soort_cnt.values)*0.01, i, f'{v:,}', va='center', fontsize=14)
#     ax.set_xlabel('Aantal deklaag-rijen', fontsize=14)
#     ax.set_title('Verdeling deklaagsoorten', fontweight='bold', fontsize=16)


# def plot_aanlegjaar(ax):
#     ax.bar(jaar_cnt.index.astype(int), jaar_cnt.values,
#            color=plt.cm.viridis(np.linspace(0.2, 0.9, len(jaar_cnt))), edgecolor='white')
#     ax.set_xlabel('Aanlegjaar', fontsize=14)
#     ax.set_ylabel('Aantal deklagen', fontsize=14)
#     ax.set_title(f'Verdeling aanlegjaar (mediaan: {int(jaar_cnt.index[len(jaar_cnt)//2])})',
#                  fontweight='bold', fontsize=16)
#     ax.tick_params(axis='x', rotation=45)


# def plot_n_per_hecto(ax):
#     ax.bar(n_dek.index.astype(int), n_dek.values, color='#9C27B0', edgecolor='white')
#     for i, v in zip(n_dek.index, n_dek.values):
#         ax.text(i, v + max(n_dek.values)*0.01, f'{v:,}', ha='center', fontsize=14)
#     ax.set_xlabel('Aantal deklagen op één hectopunt', fontsize=14)
#     ax.set_ylabel('Aantal hectopunten', fontsize=14)
#     ax.set_title('Verdeling aantal deklagen per hectopunt', fontweight='bold', fontsize=16)


# def plot_kaart_aantal(ax):
#     gdf = gpd.GeoDataFrame(
#         df[['wegnr_hmp', 'aantal_deklagen', 'hectopunt_geometry']].copy(),
#         geometry='hectopunt_geometry', crs='EPSG:28992'
#     )
#     geen = gdf[gdf['aantal_deklagen'].isna()]
#     wel  = gdf[gdf['aantal_deklagen'].notna()].copy()
#     wel['aantal_deklagen'] = wel['aantal_deklagen'].astype(int)
#     vmin = wel['aantal_deklagen'].quantile(0.20) if len(wel) else 0
#     vmax = wel['aantal_deklagen'].quantile(0.90) if len(wel) else 1
#     geen.plot(ax=ax, color='#e0e0e0', markersize=3,
#               label=f'Geen deklaag ({len(geen):,})')
#     wel.plot(ax=ax, column='aantal_deklagen', cmap='RdYlGn_r',
#              vmin=vmin, vmax=vmax, markersize=14, legend=True,
#              legend_kwds={'label': f'Aantal deklagen / hectopunt (geclipt {vmin:.0f}–{vmax:.0f})',
#                           'shrink': 0.6})
#     ax.set_title(f'Kaart: aantal deklagen per hectopunt — '
#                  f'{len(wel):,}/{len(gdf):,} gekoppeld '
#                  f'({len(wel)/len(gdf)*100:.1f}%) — groen = weinig info, rood = veel info',
#                  fontweight='bold', fontsize=15)
#     ax.set_axis_off()
#     ax.legend(loc='upper left', fontsize=12)


# def plot_kaart_dekking(ax):
#     gdf = gpd.GeoDataFrame(
#         df[['wegnr_hmp', 'deklaagsoort', 'hectopunt_geometry']].copy(),
#         geometry='hectopunt_geometry', crs='EPSG:28992'
#     )
#     gdf['heeft_deklaag'] = gdf['deklaagsoort'].notna()
#     gdf[~gdf['heeft_deklaag']].plot(ax=ax, color='#e0e0e0', markersize=3,
#                                      label=f'Geen deklaag ({(~gdf["heeft_deklaag"]).sum():,})')
#     gdf[gdf['heeft_deklaag']].plot(ax=ax, color='#2196F3', markersize=8,
#                                     label=f'Met deklaag ({gdf["heeft_deklaag"].sum():,})')
#     ax.set_title(f'Deklaag-dekking per hectopunt — '
#                  f'{gdf["heeft_deklaag"].sum():,}/{len(gdf):,} '
#                  f'({gdf["heeft_deklaag"].mean()*100:.1f}%)',
#                  fontsize=15, fontweight='bold')
#     ax.set_axis_off()
#     ax.legend(loc='upper right', fontsize=12)


# # (naam, plot_fn, figsize_4K)
# plot_specs = [
#     ('per_snelweg',     plot_per_snelweg,    (32, 14)),
#     ('deklaagsoort',    plot_deklaagsoort,   (32, 14)),
#     ('aanlegjaar',      plot_aanlegjaar,     (32, 12)),
#     ('n_per_hectopunt', plot_n_per_hecto,    (32, 12)),
#     ('kaart_aantal',    plot_kaart_aantal,   (38, 22)),
#     ('kaart_dekking',   plot_kaart_dekking,  (38, 22)),
# ]

# # 4K losse PNG's — figsize × dpi=200 → boven 4K resolutie
# for naam, plot_fn, fs in plot_specs:
#     fig, ax = plt.subplots(figsize=fs)
#     plot_fn(ax)
#     fig.tight_layout()
#     pad = OUTPUT_DIR / f'deklagen_{naam}_4k.png'
#     fig.savefig(pad, dpi=200, bbox_inches='tight')
#     print(f'Opgeslagen (4K): {pad}')
#     plt.show()
#     plt.close(fig)

# print(f'\nPer snelweg:')
# print(per_weg.sort_values('dekking_%', ascending=False).to_string())
# print(f'\nDeklaagsoort top:')
# print(soort_cnt.to_string())
# print(f'\nAanlegjaar bereik: {int(jaar_cnt.index.min())} – {int(jaar_cnt.index.max())}, '
#       f'totaal {jaar_cnt.sum():,} aanlegmomenten')

In [ ]:
# import matplotlib.pyplot as plt
# import geopandas as gpd

# gdf = gpd.GeoDataFrame(
#     df[['wegnr_hmp', 'deklaagsoort', 'aantal_deklagen', 'hectopunt_geometry']].copy(),
#     geometry='hectopunt_geometry', crs='EPSG:28992'
# )
# gdf['heeft_deklaag'] = gdf['deklaagsoort'].notna()

# fig, ax = plt.subplots(figsize=(13, 13))
# gdf[~gdf['heeft_deklaag']].plot(ax=ax, color='#e0e0e0', markersize=2,
#                                  label=f'Geen deklaag ({(~gdf["heeft_deklaag"]).sum():,})')
# gdf[gdf['heeft_deklaag']].plot(ax=ax, color='#2196F3', markersize=4,
#                                 label=f'Met deklaag ({gdf["heeft_deklaag"].sum():,})')
# ax.set_title(f'Deklaag-dekking per hectopunt — '
#              f'{gdf["heeft_deklaag"].sum():,}/{len(gdf):,} '
#              f'({gdf["heeft_deklaag"].mean()*100:.1f}%)',
#              fontsize=13, fontweight='bold')
# ax.set_axis_off()
# ax.legend(loc='upper right', fontsize=11)
# plt.tight_layout()
# plt.show()

In [ ]:
df.columns

In [ ]:
# import matplotlib.pyplot as plt
# import geopandas as gpd

# OUTPUT_DIR = Path('Output')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# def telcsv(s):
#     return 0 if pd.isna(s) else len([v for v in str(s).split(',') if v.strip()])


# df['_n_hoek']         = df['draaihoek'].apply(telcsv)
# df['_n_visibility']   = df['visibility_2025'].apply(telcsv)
# df['_n_inspecties']   = df['aantal_inspecties'].fillna(0).astype(int)
# df['_n_deklagen']     = df['aantal_deklagen'].fillna(0).astype(int)

# df['_info_basis']      = df['_n_hoek'] + df['_n_visibility'] + df['_n_inspecties']
# df['_info_uitgebreid'] = df['_info_basis'] + df['_n_deklagen']

# gdf = gpd.GeoDataFrame(
#     df[['wegnr_hmp', '_info_basis', '_info_uitgebreid', 'hectopunt_geometry']].copy(),
#     geometry='hectopunt_geometry', crs='EPSG:28992'
# )


# def plot_info(ax, kolom, titel):
#     geen = gdf[gdf[kolom] == 0]
#     wel  = gdf[gdf[kolom] >  0]
#     vmin = wel[kolom].quantile(0.20)
#     vmax = wel[kolom].quantile(0.90)
#     geen.plot(ax=ax, color='#e0e0e0', markersize=3,
#               label=f'Geen info ({len(geen):,})')
#     wel.plot(ax=ax, column=kolom, cmap='RdYlGn_r',
#              vmin=vmin, vmax=vmax, markersize=14, legend=True,
#              legend_kwds={'label': f'Datapunten / hectopunt (geclipt {vmin:.0f}–{vmax:.0f})',
#                           'shrink': 0.6})
#     ax.set_title(f'{titel} — verdeling: '
#                  f'min={wel[kolom].min()}, mediaan={wel[kolom].median():.0f}, '
#                  f'p90={vmax:.0f}, max={wel[kolom].max()}',
#                  fontweight='bold', fontsize=14)
#     ax.set_axis_off()
#     ax.legend(loc='upper left', fontsize=12)


# plot_specs = [
#     ('basis',      '_info_basis',
#      'Informatie-omvang per hectopunt — hoek + inspecties + visibility (2025)'),
#     ('uitgebreid', '_info_uitgebreid',
#      'Informatie-omvang per hectopunt — incl. deklagen'),
# ]

# # 4K losse PNG's — 38" × 22" @ dpi=200 → ~7600 × 4400 px per plot
# for naam, kolom, titel in plot_specs:
#     fig, ax = plt.subplots(figsize=(38, 22))
#     plot_info(ax, kolom, titel)
#     fig.tight_layout()
#     pad = OUTPUT_DIR / f'datadekking_{naam}_4k.png'
#     fig.savefig(pad, dpi=200, bbox_inches='tight')
#     print(f'Opgeslagen (4K): {pad}')
#     plt.show()
#     plt.close(fig)

# df = df.drop(columns=['_n_hoek', '_n_visibility', '_n_inspecties', '_n_deklagen',
#                       '_info_basis', '_info_uitgebreid'])

In [ ]:
df.head(10).T